### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

## vector store
from langchain_community.vectorstores import Chroma

##utilities
import numpy as np
from typing import List, Tuple

/home/divyanshu/Desktop/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### 1. Sample Data

In [4]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [5]:
##save the documents to text files
import tempfile
temp_dir=tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)
print(f"Documents saved to: {temp_dir}")

Documents saved to: /tmp/tmpj4duvzmf


### 2. Document Loading

In [6]:
from langchain_community.document_loaders import DirectoryLoader
#Load documents from the temporary directory
loader = DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls= TextLoader,
    loader_kwargs={"encoding":"utf-8"}
)

In [7]:
documents =loader.load()
print(f"Loaded {len(documents)} documents.")
print("Sample Document:")
print(documents[0].page_content[:500])

Loaded 3 documents.
Sample Document:

    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    


### Document Splitting

In [8]:
# Intialize text splitter
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function= len,
    separators=[" "]
)
chunks= text_splitter.split_documents(documents)
print(f"Total Chunks Created: {len(chunks)} from {len(documents)} documents.")
print("Sample Chunk:")
print(chunks[0].page_content[:500])
print(f"Chunk 1 Metadata: {chunks[0].metadata}")
print(chunks[1].page_content[:500])
print(f"Chunk 2 Metadata: {chunks[1].metadata}")


Total Chunks Created: 5 from 3 documents.
Sample Chunk:
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
Chunk 1 Metadata: {'source': '/tmp/tmpj4duvzmf/doc_2.txt'}
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds pat

### Embedding Models

In [9]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [10]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7f8a46f69010>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7f8a46f69940>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base='https://models.inference.ai.azure.com', openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [11]:
sample_query="Machine learning"
vector = embeddings.embed_query(sample_query)
vector

[-0.00931549072265625,
 -0.0023517608642578125,
 -0.0008411407470703125,
 -0.021881103515625,
 0.042205810546875,
 0.01093292236328125,
 0.007640838623046875,
 0.039642333984375,
 -0.0195770263671875,
 0.022491455078125,
 -0.005767822265625,
 -0.05035400390625,
 -0.031829833984375,
 -0.0201568603515625,
 0.01313018798828125,
 0.0184783935546875,
 -0.00812530517578125,
 -0.03509521484375,
 0.0207366943359375,
 0.006153106689453125,
 0.0164337158203125,
 0.019775390625,
 0.036834716796875,
 -0.02386474609375,
 0.037994384765625,
 -0.015228271484375,
 0.034454345703125,
 0.03131103515625,
 0.00824737548828125,
 -0.0286865234375,
 -0.02838134765625,
 -0.0188446044921875,
 -0.042205810546875,
 -0.00482940673828125,
 0.031707763671875,
 -0.0132598876953125,
 -0.008697509765625,
 0.0355224609375,
 -0.048675537109375,
 0.016845703125,
 -0.0246429443359375,
 -0.0269927978515625,
 0.0163726806640625,
 0.07177734375,
 -0.0157012939453125,
 -0.0249481201171875,
 -0.042877197265625,
 -0.04385375976

### INitialize the ChromaDB Store and Store the chunks in Vector Representation

In [12]:
## Create a chromadb directory
persist_dir ="./chromadb"
vectorstore = Chroma.from_documents(
    documents=chunks,
    persist_directory=persist_dir,
    embedding=embeddings,
    collection_name="chroma_documents"
)
print(f"Created {vectorstore._collection.count()} vectors")
print(f"Stored at {persist_dir}")

Created 5 vectors
Stored at ./chromadb


### Similarity Search

In [14]:
sample_query="What is NLP"
similar_docs = vectorstore.similarity_search(sample_query,k=3)
similar_docs

[Document(metadata={'source': '/tmp/tmpj4duvzmf/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': '/tmp/tmpj4duvzmf/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Ne

In [15]:
sample_query="What is deep learning"
similar_docs = vectorstore.similarity_search(sample_query,k=3)
similar_docs

[Document(metadata={'source': '/tmp/tmpj4duvzmf/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': '/tmp/tmpj4duvzmf/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning us

In [16]:
print(f"Sample Query: {sample_query}")
print(f"Top {len(similar_docs)} Similar Chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\nChunk {i+1} Metadata: {doc.metadata}")
    print(f"Chunk {i+1} Content: {doc.page_content[:500]}...")

Sample Query: What is deep learning
Top 3 Similar Chunks:

Chunk 1 Metadata: {'source': '/tmp/tmpj4duvzmf/doc_1.txt'}
Chunk 1 Content: Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers...

Chunk 2 Metadata: {'source': '/tmp/tmpj4duvzmf/doc_0.txt'}
Chunk 2 Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and 

In [17]:
results_scores = vectorstore.similarity_search_with_score(sample_query,k=3)
results_scores

[(Document(metadata={'source': '/tmp/tmpj4duvzmf/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  0.7544082403182983),
 (Document(metadata={'source': '/tmp/tmpj4duvzmf/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learnin

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)